# Qwen3-VL-8B-Instruct — проверка на RunPod A40

`Qwen/Qwen3-VL-8B-Instruct` — свежая открытая VL-модель (не gated, `apache-2.0`, токен не нужен).
В отличие от Sherkala/KazLLM казахский официально не заявлен и не является фокусом модели — это скорее
ориентир "насколько далеко общая сильная VL-модель уезжает от специализированной Qolda" на kk/OCR,
а не прямой конкурент по языку.

Плюс у неё, в отличие от Qolda, **штатный tool calling** через `transformers` — стоит один раз проверить,
работает ли он из коробки, для будущего этапа 3.

**Важно:** нужен `transformers>=4.57.0` — новее версии `4.55.0`, закреплённой в ноутбуках Qolda/Sherkala/KazLLM.
Архитектура `qwen3_vl` не custom code (в отличие от Qolda), так что конфликтов вроде `AttributeError:
'InternVLChatModel' object has no attribute 'all_tied_weights_keys'` здесь не ожидается — но если на этом
поде уже стоит `4.55.0` из прошлых ноутбуков, эта ячейка её обновит именно для текущего ядра.

In [ ]:
# --- 0. Проверка GPU ---
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram total GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
    print('vram free GB:', round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 2**30, 1))
else:
    raise SystemExit('GPU недоступна в этом контейнере.')

In [ ]:
# --- Кэш в /workspace (переживает рестарт пода, / — всего 5GB) ---
import os
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.makedirs('/workspace/hf_cache', exist_ok=True)
os.makedirs('/workspace/test_docs', exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# --- Зависимости ---
# Не gated -> без токена. Нужна конкретная версия >=4.57.0 (Qwen3-VL добавлена туда),
# ставим её точно, не -U, чтобы результат был воспроизводим.
%pip install -q "transformers==4.57.0" accelerate qwen-vl-utils pillow hf_transfer

In [ ]:
import transformers
print('transformers version:', transformers.__version__)
assert transformers.__version__ == '4.57.0', 'Если версия другая — перезапусти ядро (Kernel -> Restart) и выполни заново.'

## Загрузка модели

`Qwen3VLForConditionalGeneration` + `AutoProcessor` (обрабатывает и текст, и изображения одним объектом,
в отличие от Qolda, где токенайзер и image-preprocessing собирались вручную).

In [ ]:
import time
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

MODEL_ID = 'Qwen/Qwen3-VL-8B-Instruct'

t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).eval().cuda()
print(f'Загружено за {time.time()-t0:.1f}s')
print('VRAM занято, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

## Тест 1 — языки (kk/ru/en)

Те же промпты, что в ноутбуках Qolda/Sherkala/KazLLM — для прямого сравнения бок о бок.

In [ ]:
GEN_KW = dict(max_new_tokens=512, do_sample=False)

def ask_text(question):
    messages = [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KW)
    new_tokens = out[0][inputs['input_ids'].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

prompts = {
    'kk': 'Қазақстанның бас қаласы қандай және неге астана болып таңдалды? Қысқаша жауап бер.',
    'ru': 'Объясни своими словами, чем отличается ИП от ТОО в Казахстане. Кратко.',
    'en': 'In two sentences, explain what a vector database is used for in RAG systems.',
}

for lang, q in prompts.items():
    t0 = time.time()
    answer = ask_text(q)
    dt = time.time() - t0
    print(f'=== {lang} ({dt:.1f}s) ===')
    print('Q:', q)
    print('A:', answer)
    print()

**На что смотреть:** казахский у Qwen3-VL официально не заявлен — вероятна более слабая грамматика/лексика,
чем у Qolda/Sherkala/KazLLM. Это ожидаемая точка сравнения: показывает разницу между "сильная общая модель"
и "модель, специально дообученная под язык".

## Тест 2 — зрение / OCR документа

Используем тот же тестовый файл, что и в `01_qolda_check.ipynb` (`/workspace/test_docs/sample.png`) —
если его там ещё нет, ячейка сгенерирует тот же синтетический документ.

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

SAMPLE_PATH = Path('/workspace/test_docs/sample.png')

if not SAMPLE_PATH.exists():
    print('Реального скана нет — генерирую синтетический тестовый документ (тот же, что у Qolda).')
    img = Image.new('RGB', (1000, 500), 'white')
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 28)
    except Exception:
        font = ImageFont.load_default()
    lines = [
        'АНЫҚТАМА / СПРАВКА',
        'Тегі: Ахметов А.Б.',
        'Мекенжайы: Алматы қаласы, Абай көшесі 10',
        'Дата выдачи: 06.08.2026',
        'Номер документа: 45231-KZ',
    ]
    for i, line in enumerate(lines):
        draw.text((40, 40 + i * 60), line, fill='black', font=font)
    img.save(SAMPLE_PATH)
    print('Синтетический документ сохранён:', SAMPLE_PATH)
else:
    print('Использую существующий файл:', SAMPLE_PATH)

img = Image.open(SAMPLE_PATH)
print('Размер:', img.size)
img

In [ ]:
# Qwen3-VL принимает изображение прямо в сообщении через AutoProcessor — без ручного
# тайлинга 448x448, которым занимались вручную для Qolda. Модель сама решает разрешение/патчи.
def ask_with_image(image_path, question):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': question},
        ],
    }]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KW)
    new_tokens = out[0][inputs['input_ids'].shape[-1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

t0 = time.time()
answer = ask_with_image(
    SAMPLE_PATH,
    'Прочитай весь текст на изображении дословно, ничего не придумывай. Если что-то нечитаемо — так и скажи.',
)
dt = time.time() - t0
print(f'--- Ответ ({dt:.1f}s) ---')
print(answer)

**Сравнить с результатом Qolda из `01_qolda_check.ipynb`:** совпадает ли качество распознавания казахского/русского
текста на одном и том же синтетическом документе. Если Qwen3-VL читает не хуже — это довод в пользу более
универсальной модели вместо узкоспециализированной Qolda (учитывая, что зрение у Qolda было главным риском плана).

In [ ]:
# --- Доп. вопрос по документу (проверка понимания, а не только распознавания) ---
answer2 = ask_with_image(SAMPLE_PATH, 'Какая дата указана в документе и какой у него номер?')
print(answer2)

## Тест 3 — tool calling

Не входит в scope этого этапа, но раз модель заявляет штатную поддержку — быстрый прогон для этапа 3.

In [ ]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Получить текущую погоду в городе',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string', 'description': 'Название города'}},
            'required': ['city'],
        },
    },
}]

messages = [{'role': 'user', 'content': [{'type': 'text', 'text': 'Қандай ауа райы Алматыда қазір?'}]}]

try:
    inputs = processor.apply_chat_template(
        messages, tools=tools, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KW)
    new_tokens = out[0][inputs['input_ids'].shape[-1]:]
    print(processor.decode(new_tokens, skip_special_tokens=True).strip())
except Exception as e:
    print('apply_chat_template не принял tools= напрямую:', e)

## Замер VRAM

In [ ]:
print('VRAM allocated, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))
print('VRAM reserved,  GB:', round(torch.cuda.memory_reserved() / 2**30, 2))
print('VRAM peak,      GB:', round(torch.cuda.max_memory_allocated() / 2**30, 2))

In [ ]:
# Освободить память, если на этом поде дальше запускается что-то ещё.
del model
torch.cuda.empty_cache()
print('VRAM после очистки, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

## Дальше по плану

Сравнить результаты (языки, зрение, VRAM, latency, tool calling) с Qolda в `docs/findings.md`.
Ключевой вопрос: оправдывает ли себя специализация Qolda под казахский, или общая сильная VL-модель
уровня Qwen3-VL закрывает задачу не хуже — с бонусом в виде рабочего tool calling.